# 40 Setup Landing Zone

# Ingesta incremental

En un pipeline real, los datos llegan continuamente a una *landing zone* en object storage
(S3, ADLS o GCS): un sistema externo deposita archivos y nuestro trabajo es ingerirlos a la
capa Bronze **sin duplicar y sin perder ninguno**.

En Databricks Free Edition no tenemos acceso a object storage externo, así que un
**Volume de Unity Catalog** cumple ese rol: es un directorio gobernado por el catálogo,
accesible como filesystem normal.

Este notebook:

* Crea el schema y el volume que usarán los notebooks 41, 42 y 43.
* Define las funciones que **simulan** al sistema externo: generan lotes de archivos
  JSON/CSV con eventos de calificaciones de películas de MovieLens
  (`user_id`, `movie_id`, `rating`, `ts`).
.

In [ ]:
# Celda de configuración (primera celda de código del notebook)
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
VOLUME  = "landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Rutas base: cada notebook usa su propio subdirectorio para evitar colisiones
VOL_BASE     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
LANDING_AL   = f"{VOL_BASE}/autoloader"    # notebook 41 (Auto Loader)
LANDING_CI   = f"{VOL_BASE}/copyinto"      # notebook 42 (COPY INTO)
LANDING_JSON = f"{VOL_BASE}/json_semi"     # notebook 43 (JSON / VARIANT)
CHK_BASE     = f"{VOL_BASE}/chk"           # checkpoints y schema locations
CHK_41       = f"{CHK_BASE}/41"
CHK_43       = f"{CHK_BASE}/43"

print(f"Trabajando en {CATALOG}.{SCHEMA}")
print(f"Zona de aterrizaje: {VOL_BASE}")

NameError: name 'spark' is not defined

> **Antes de ejecutar:** Esta celda borra y recrea los subdirectorios de trabajo. Si la ejecutas dos veces seguidas, ¿la segunda vez falla porque los directorios ya existen, o pasa sin error? ¿Qué implica eso para *Restart & Run All*?

In [ ]:
# Limpieza idempotente: cada corrida del laboratorio arranca desde cero.
# La incrementalidad se demuestra DENTRO de una misma ejecución
# (lote 1 -> ingesta -> lote 2 -> ingesta), así el notebook siempre es reproducible
# con Restart & Run All.
for sub in [LANDING_AL, LANDING_CI, LANDING_JSON, CHK_BASE]:
    dbutils.fs.rm(sub, recurse=True)
    dbutils.fs.mkdirs(sub)

print("Zona de aterrizaje y checkpoints reiniciados.")

## El "productor" de archivos

Las funciones de abajo simulan al sistema externo que deposita lotes en la landing zone:

* `generar_lote_ratings(...)` escribe un archivo `lote_XX.json` (JSON Lines, un evento por línea).
  * `columnas_extra=True` agrega el campo `device` — lo usaremos para provocar **evolución de esquema**.
  * `anidado=True` genera la versión con structs y arrays (`user`, `movie.genres`) — para el notebook 43.
* `generar_lote_csv(...)` escribe la versión CSV con encabezado — para COPY INTO (notebook 42).

Nota: los Volumes se pueden leer/escribir con la API estándar de Python (`open`, `os`),
como si fueran un disco local.

In [ ]:
import json, csv, os, random
from datetime import datetime, timedelta

DEVICES  = ["mobile", "web", "tv"]
GENEROS  = ["Action", "Comedy", "Drama", "Horror", "Romance", "Sci-Fi", "Thriller", "Animation"]
PAISES   = ["CR", "MX", "CO", "AR", "CL", "PE"]
PLANES   = ["basic", "standard", "premium"]
CIUDADES = {"CR": "San Jose", "MX": "CDMX", "CO": "Bogota",
            "AR": "Buenos Aires", "CL": "Santiago", "PE": "Lima"}

def _evento_plano(rnd, base_ts, i, columnas_extra):
    ev = {
        "user_id":  rnd.randint(1, 1000),
        "movie_id": rnd.randint(1, 2000),
        "rating":   rnd.randint(1, 10) * 0.5,
        "ts":       (base_ts + timedelta(seconds=rnd.randint(0, 86_399))).isoformat(),
    }
    if columnas_extra:
        ev["device"] = rnd.choice(DEVICES)
    return ev

def _evento_anidado(rnd, base_ts, i, columnas_extra):
    country = rnd.choice(PAISES)
    user = {"id": rnd.randint(1, 1000), "country": country, "plan": rnd.choice(PLANES)}
    if columnas_extra:
        user["city"] = CIUDADES[country]
    return {
        "event_id": i,
        "user":  user,
        "movie": {"id": rnd.randint(1, 2000),
                  "genres": rnd.sample(GENEROS, rnd.randint(1, 3))},
        "rating": rnd.randint(1, 10) * 0.5,
        "ts":     (base_ts + timedelta(seconds=rnd.randint(0, 86_399))).isoformat(),
    }

def generar_lote_ratings(destino, lote, n=500, columnas_extra=False, anidado=False, seed=None):
    """Escribe un archivo lote_XX.json (JSON Lines) simulando a un sistema externo."""
    rnd = random.Random(seed if seed is not None else lote)   # seed = lote -> reproducible
    base_ts = datetime(2026, 7, 1) + timedelta(days=lote)
    ruta = f"{destino}/lote_{lote:02d}.json"
    os.makedirs(destino, exist_ok=True)
    gen = _evento_anidado if anidado else _evento_plano
    with open(ruta, "w") as f:
        for i in range(n):
            f.write(json.dumps(gen(rnd, base_ts, i + lote * 100_000, columnas_extra)) + "\n")
    print(f"Lote {lote}: {n} eventos escritos en {ruta}")
    return ruta

def generar_lote_csv(destino, lote, n=500, seed=None):
    """Versión CSV (con encabezado) del mismo productor — para COPY INTO."""
    rnd = random.Random(seed if seed is not None else lote)
    base_ts = datetime(2026, 7, 1) + timedelta(days=lote)
    ruta = f"{destino}/lote_{lote:02d}.csv"
    os.makedirs(destino, exist_ok=True)
    with open(ruta, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["user_id", "movie_id", "rating", "ts"])
        for _ in range(n):
            ev = _evento_plano(rnd, base_ts, 0, False)
            w.writerow([ev["user_id"], ev["movie_id"], ev["rating"], ev["ts"]])
    print(f"Lote {lote}: {n} filas escritas en {ruta}")
    return ruta

In [ ]:
# Generar un lote pequeño en un directorio aparte ("Smoke test") para probar la ingesta de datos sin tener que esperar a que se generen 500 eventos. Esto es útil para depuración y pruebas rápidas.
smoke_dir = f"{VOL_BASE}/smoke"
generar_lote_ratings(smoke_dir, lote=0, n=5)
display(dbutils.fs.ls(smoke_dir))

SyntaxError: invalid syntax (2602085821.py, line 1)

In [ ]:
# Prueba de humo (2/2): lo leemos con Spark y borramos el directorio
df_smoke = spark.read.json(f"{smoke_dir}/lote_00.json")
df_smoke.printSchema()
display(df_smoke)

dbutils.fs.rm(smoke_dir, recurse=True)
print("Prueba de humo completada; directorio smoke/ eliminado.")